# YOLOv26n Sequence Model Evaluation

Notebook-first evaluation for the PAPI lamp model. This evaluates detection, red/white classification, per-lamp tracking state, and red/white transition events. Transition detection is derived from tracked lamp states; it is not a YOLO class.


## Policy

For this project, deep-learning and machine-learning work should be run from notebooks. This notebook replaces the standalone evaluation script and keeps the evaluation logic inline for inspection and reruns.


In [ ]:
from pathlib import Path
import json

REPO_ROOT = Path.cwd()
WEIGHTS = REPO_ROOT / "runs" / "papi" / "yolo26n_sequence_red_white_safe" / "weights" / "best.pt"
DATASET_ROOT = REPO_ROOT / "data" / "datasets" / "papi_lamp_sequences"
SPLIT_ROOT = DATASET_ROOT / "yolo26n_combined"
RAW_OUTPUT_DIR = REPO_ROOT / "runs" / "papi" / "yolo26n_sequence_red_white_safe" / "proper_eval"
TOP4_OUTPUT_DIR = REPO_ROOT / "runs" / "papi" / "yolo26n_sequence_red_white_safe" / "proper_eval_top4"

assert WEIGHTS.exists(), WEIGHTS
assert SPLIT_ROOT.exists(), SPLIT_ROOT


## Evaluation Implementation

The next cell contains the full evaluation implementation. It runs YOLO predictions once per split, sweeps confidence thresholds, optionally keeps only the top four predictions per frame, matches detections to ground truth at IoU 0.5, assigns detections to lamp tracks, and compares transition events with a configurable frame tolerance.


In [ ]:
"""Evaluate detector, class labels, per-lamp tracking, and transitions on PAPI splits."""

from __future__ import annotations

import argparse
import csv
import json
import tempfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import yaml
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO

from papi.projection import DEFAULT_CONVENTION, ProjectionConvention
from papi.tracking import (
    YoloDetection,
    assign_frame_tracks,
    detect_transitions,
    read_yolo_detections,
)

REPO_ROOT = Path.cwd()
CLASS_NAMES = {0: "papi_light_red", 1: "papi_light_white"}


@dataclass(frozen=True)
class Prediction:
    row_index: int
    class_id: int
    confidence: float
    cx: float
    cy: float
    w: float
    h: float
    image_width: int
    image_height: int

    @property
    def xyxy_px(self) -> tuple[float, float, float, float]:
        half_w = self.w * self.image_width / 2
        half_h = self.h * self.image_height / 2
        cx_px = self.cx * self.image_width
        cy_px = self.cy * self.image_height
        return (cx_px - half_w, cy_px - half_h, cx_px + half_w, cy_px + half_h)


def _load_yaml(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def _load_projection(path: Path) -> ProjectionConvention:
    if not path.exists():
        return DEFAULT_CONVENTION
    return ProjectionConvention.from_dict(_load_yaml(path)["convention"])


def _read_split(path: Path) -> list[Path]:
    return [Path(line.strip()) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def _metadata_index(dataset_root: Path) -> dict[Path, tuple[str, Path, dict[str, Any]]]:
    index: dict[Path, tuple[str, Path, dict[str, Any]]] = {}
    for metadata_path in dataset_root.glob("*/*/metadata.csv"):
        video_dir = metadata_path.parent
        with metadata_path.open(newline="", encoding="utf-8") as fh:
            for row in csv.DictReader(fh):
                image_path = (video_dir / str(row["image"])).resolve()
                index[image_path] = (video_dir.name, video_dir, row)
    return index


def _gt_box(det: YoloDetection) -> tuple[float, float, float, float]:
    half_w = det.w * det.image_width / 2
    half_h = det.h * det.image_height / 2
    return (det.cx_px - half_w, det.cy_px - half_h, det.cx_px + half_w, det.cy_px + half_h)


def _iou(a: tuple[float, float, float, float], b: tuple[float, float, float, float]) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    inter_w = max(0.0, ix2 - ix1)
    inter_h = max(0.0, iy2 - iy1)
    inter = inter_w * inter_h
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    return inter / max(area_a + area_b - inter, 1e-9)


def _match_predictions(
    gt: list[YoloDetection],
    pred: list[Prediction],
    iou_threshold: float,
) -> tuple[list[tuple[int, int, float]], set[int], set[int]]:
    if not gt or not pred:
        return [], set(), set()
    ious = np.array(
        [[_iou(_gt_box(gt_box), pred_box.xyxy_px) for pred_box in pred] for gt_box in gt],
        dtype=float,
    )
    gt_indices, pred_indices = linear_sum_assignment(-ious)
    matches: list[tuple[int, int, float]] = []
    matched_gt: set[int] = set()
    matched_pred: set[int] = set()
    for gt_i, pred_i in zip(gt_indices, pred_indices, strict=True):
        score = float(ious[gt_i, pred_i])
        if score < iou_threshold:
            continue
        matches.append((gt_i, pred_i, score))
        matched_gt.add(gt_i)
        matched_pred.add(pred_i)
    return matches, matched_gt, matched_pred


def _prediction_to_yolo(prediction: Prediction) -> YoloDetection:
    return YoloDetection(
        row_index=prediction.row_index,
        class_id=prediction.class_id,
        cx=prediction.cx,
        cy=prediction.cy,
        w=prediction.w,
        h=prediction.h,
        image_width=prediction.image_width,
        image_height=prediction.image_height,
    )


def _event_key(row: dict[str, str]) -> tuple[str, str, str, str, str]:
    return (
        row["video_id"],
        row["physical_lamp_id"],
        row["from_frame_index"],
        row["to_frame_index"],
        row["transition_type"],
    )


def _match_transition_events(
    gt_events: set[tuple[str, str, str, str, str]],
    pred_events: set[tuple[str, str, str, str, str]],
    frame_tolerance: int,
) -> tuple[int, int, int]:
    unmatched_pred = set(pred_events)
    true_positives = 0
    for gt_event in sorted(gt_events):
        candidates = [
            pred_event
            for pred_event in unmatched_pred
            if pred_event[0] == gt_event[0]
            and pred_event[1] == gt_event[1]
            and pred_event[4] == gt_event[4]
            and abs(int(pred_event[2]) - int(gt_event[2])) <= frame_tolerance
            and abs(int(pred_event[3]) - int(gt_event[3])) <= frame_tolerance
        ]
        if not candidates:
            continue
        best = min(
            candidates,
            key=lambda pred_event: abs(int(pred_event[2]) - int(gt_event[2]))
            + abs(int(pred_event[3]) - int(gt_event[3])),
        )
        unmatched_pred.remove(best)
        true_positives += 1
    false_positives = len(unmatched_pred)
    false_negatives = len(gt_events) - true_positives
    return true_positives, false_positives, false_negatives


def _run_predictions(
    model_path: Path,
    image_paths: list[Path],
    min_conf: float,
    imgsz: int,
    batch: int,
    device: str,
) -> dict[Path, list[Prediction]]:
    model = YOLO(str(model_path))
    predictions: dict[Path, list[Prediction]] = {}
    chunk_size = max(batch * 4, 1)
    for start in range(0, len(image_paths), chunk_size):
        chunk = image_paths[start : start + chunk_size]
        with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as fh:
            fh.write("\n".join(str(path) for path in chunk))
            fh.write("\n")
            source_file = Path(fh.name)
        try:
            results = model.predict(
                source=str(source_file),
                conf=min_conf,
                imgsz=imgsz,
                batch=batch,
                device=device,
                stream=True,
                verbose=False,
            )
            for image_path, result in zip(chunk, results, strict=True):
                height, width = result.orig_shape
                rows: list[Prediction] = []
                for row_index, box in enumerate(result.boxes, 1):
                    cls = int(box.cls.item())
                    if cls not in CLASS_NAMES:
                        continue
                    cx, cy, w, h = (float(value) for value in box.xywhn[0].tolist())
                    rows.append(
                        Prediction(
                            row_index=row_index,
                            class_id=cls,
                            confidence=float(box.conf.item()),
                            cx=cx,
                            cy=cy,
                            w=w,
                            h=h,
                            image_width=width,
                            image_height=height,
                        )
                    )
                predictions[image_path.resolve()] = rows
        finally:
            source_file.unlink(missing_ok=True)
    return predictions


def evaluate_threshold(
    *,
    split_name: str,
    image_paths: list[Path],
    metadata: dict[Path, tuple[str, Path, dict[str, Any]]],
    predictions: dict[Path, list[Prediction]],
    airport_config: dict[str, Any],
    projection_convention: ProjectionConvention,
    threshold: float,
    max_predictions_per_frame: int | None,
    iou_threshold: float,
    transition_frame_tolerance: int,
    projection_max_distance_px: float,
) -> dict[str, Any]:
    counts: Counter[str] = Counter()
    confusion: Counter[str] = Counter()
    iou_sum = 0.0
    pred_track_rows: list[dict[str, str]] = []
    gt_track_rows_by_video: dict[str, list[dict[str, str]]] = defaultdict(list)
    split_files_by_video: dict[str, set[str]] = defaultdict(set)
    frame_examples: list[dict[str, Any]] = []

    for image_path in image_paths:
        resolved = image_path.resolve()
        video_id, video_dir, row = metadata[resolved]
        split_files_by_video[video_id].add(str(row["file"]))
        image_width = int(row["image_width"])
        image_height = int(row["image_height"])
        gt = read_yolo_detections(video_dir / str(row["label"]), image_width, image_height)
        pred = [item for item in predictions.get(resolved, []) if item.confidence >= threshold]
        if max_predictions_per_frame is not None:
            pred = sorted(pred, key=lambda item: item.confidence, reverse=True)[
                :max_predictions_per_frame
            ]
        matches, matched_gt, matched_pred = _match_predictions(gt, pred, iou_threshold)

        counts["images"] += 1
        counts["gt_boxes"] += len(gt)
        counts["pred_boxes"] += len(pred)
        counts["matched_boxes"] += len(matches)
        counts["false_positive_boxes"] += len(pred) - len(matched_pred)
        counts["false_negative_boxes"] += len(gt) - len(matched_gt)
        counts["frames_with_four_predictions"] += int(len(pred) == 4)
        counts["frames_with_exact_box_count"] += int(len(pred) == len(gt))

        for gt_i, pred_i, score in matches:
            gt_class = gt[gt_i].class_id
            pred_class = pred[pred_i].class_id
            iou_sum += score
            counts["localized_classifications"] += 1
            counts["correct_classifications"] += int(gt_class == pred_class)
            confusion[f"{CLASS_NAMES[gt_class]}->{CLASS_NAMES[pred_class]}"] += 1

        if len(frame_examples) < 25 and (len(pred) != len(gt) or len(matches) != len(gt)):
            frame_examples.append(
                {
                    "split": split_name,
                    "video_id": video_id,
                    "file": str(row["file"]),
                    "gt_boxes": len(gt),
                    "pred_boxes": len(pred),
                    "matched_boxes": len(matches),
                }
            )

        gt_track_rows_by_video[video_id].extend(
            assign_frame_tracks(
                video_id=video_id,
                image_row=row,
                label_rel=str(row["label"]),
                detections=gt,
                airport_config=airport_config,
                projection_convention=projection_convention,
                projection_max_distance_px=projection_max_distance_px,
            )
        )
        pred_track_rows.extend(
            assign_frame_tracks(
                video_id=video_id,
                image_row=row,
                label_rel=str(row["label"]),
                detections=[_prediction_to_yolo(item) for item in pred],
                airport_config=airport_config,
                projection_convention=projection_convention,
                projection_max_distance_px=projection_max_distance_px,
            )
        )

    gt_state_by_lamp = {
        (row["video_id"], row["frame_index"], row["physical_lamp_id"]): row["state"]
        for rows in gt_track_rows_by_video.values()
        for row in rows
        if row["physical_lamp_id"]
    }
    pred_state_by_lamp = {
        (row["video_id"], row["frame_index"], row["physical_lamp_id"]): row["state"]
        for row in pred_track_rows
        if row["physical_lamp_id"]
    }
    for key, gt_state in gt_state_by_lamp.items():
        pred_state = pred_state_by_lamp.get(key)
        counts["tracked_lamp_states"] += 1
        if pred_state is None:
            counts["missing_tracked_lamp_states"] += 1
        elif pred_state == gt_state:
            counts["correct_tracked_lamp_states"] += 1
        else:
            counts["wrong_tracked_lamp_states"] += 1
            confusion[f"tracked_{gt_state}->tracked_{pred_state}"] += 1

    gt_transitions: list[dict[str, str]] = []
    for video_id, rows in gt_track_rows_by_video.items():
        allowed_files = split_files_by_video[video_id]
        gt_transitions.extend(
            row
            for row in detect_transitions(rows)
            if row["from_file"] in allowed_files and row["to_file"] in allowed_files
        )
    pred_transitions = detect_transitions(pred_track_rows)
    gt_events = {_event_key(row) for row in gt_transitions}
    pred_events = {_event_key(row) for row in pred_transitions}
    exact_transition_tp = len(gt_events & pred_events)
    exact_transition_fp = len(pred_events - gt_events)
    exact_transition_fn = len(gt_events - pred_events)
    transition_tp, transition_fp, transition_fn = _match_transition_events(
        gt_events,
        pred_events,
        transition_frame_tolerance,
    )
    pred_methods = Counter(row["assignment_method"] for row in pred_track_rows)
    pred_flags = Counter(
        flag for row in pred_track_rows for flag in row["quality_flags"].split(";") if flag
    )

    detection_precision = counts["matched_boxes"] / max(counts["pred_boxes"], 1)
    detection_recall = counts["matched_boxes"] / max(counts["gt_boxes"], 1)
    classification_accuracy = counts["correct_classifications"] / max(
        counts["localized_classifications"], 1
    )
    tracked_state_accuracy = counts["correct_tracked_lamp_states"] / max(
        counts["tracked_lamp_states"], 1
    )
    transition_precision = transition_tp / max(transition_tp + transition_fp, 1)
    transition_recall = transition_tp / max(transition_tp + transition_fn, 1)
    return {
        "threshold": threshold,
        "max_predictions_per_frame": max_predictions_per_frame,
        "iou_threshold": iou_threshold,
        "images": counts["images"],
        "detection": {
            "gt_boxes": counts["gt_boxes"],
            "pred_boxes": counts["pred_boxes"],
            "matched_boxes": counts["matched_boxes"],
            "false_positive_boxes": counts["false_positive_boxes"],
            "false_negative_boxes": counts["false_negative_boxes"],
            "precision_iou50_class_agnostic": detection_precision,
            "recall_iou50_class_agnostic": detection_recall,
            "mean_iou_for_matches": iou_sum / max(counts["matched_boxes"], 1),
            "frames_with_four_predictions_rate": counts["frames_with_four_predictions"]
            / max(counts["images"], 1),
            "frames_with_exact_box_count_rate": counts["frames_with_exact_box_count"]
            / max(counts["images"], 1),
        },
        "classification": {
            "localized_samples": counts["localized_classifications"],
            "correct": counts["correct_classifications"],
            "accuracy_on_localized_matches": classification_accuracy,
            "confusion": dict(sorted(confusion.items())),
        },
        "tracking": {
            "tracked_lamp_states": counts["tracked_lamp_states"],
            "correct_tracked_lamp_states": counts["correct_tracked_lamp_states"],
            "missing_tracked_lamp_states": counts["missing_tracked_lamp_states"],
            "wrong_tracked_lamp_states": counts["wrong_tracked_lamp_states"],
            "tracked_lamp_state_accuracy": tracked_state_accuracy,
            "pred_assignment_methods": dict(pred_methods),
            "pred_quality_flags": dict(pred_flags),
        },
        "transitions": {
            "gt_events": len(gt_events),
            "pred_events": len(pred_events),
            "frame_tolerance": transition_frame_tolerance,
            "true_positive_events": transition_tp,
            "false_positive_events": transition_fp,
            "false_negative_events": transition_fn,
            "precision": transition_precision,
            "recall": transition_recall,
            "f1": 2
            * transition_precision
            * transition_recall
            / max(transition_precision + transition_recall, 1e-9),
            "exact_frame": {
                "true_positive_events": exact_transition_tp,
                "false_positive_events": exact_transition_fp,
                "false_negative_events": exact_transition_fn,
            },
        },
        "example_frame_issues": frame_examples,
    }


def _best_by_threshold(results: list[dict[str, Any]]) -> dict[str, Any]:
    return max(
        results,
        key=lambda item: (
            item["transitions"]["f1"],
            item["tracking"]["tracked_lamp_state_accuracy"],
            item["detection"]["recall_iou50_class_agnostic"],
        ),
    )


def _write_markdown(summary: dict[str, Any], path: Path) -> None:
    lines = [
        "# PAPI YOLO26n Sequence Evaluation",
        "",
        f"- Weights: `{summary['weights']}`",
        f"- Dataset: `{summary['dataset_root']}`",
        f"- Thresholds: {', '.join(str(x) for x in summary['thresholds'])}",
        "",
        "## Best Operating Points",
        "",
    ]
    for split, split_summary in summary["splits"].items():
        best = split_summary["best_threshold"]
        lines.extend(
            [
                f"### {split}",
                "",
                f"- threshold: `{best['threshold']}`",
                "- detection: "
                f"precision {best['detection']['precision_iou50_class_agnostic']:.3f}, "
                f"recall {best['detection']['recall_iou50_class_agnostic']:.3f}, "
                f"four-prediction frames {best['detection']['frames_with_four_predictions_rate']:.3f}",
                "- classification on localized matches: "
                f"{best['classification']['accuracy_on_localized_matches']:.3f}",
                "- tracked lamp-state accuracy: "
                f"{best['tracking']['tracked_lamp_state_accuracy']:.3f} "
                f"(missing {best['tracking']['missing_tracked_lamp_states']}, "
                f"wrong {best['tracking']['wrong_tracked_lamp_states']})",
                "- transition events: "
                f"precision {best['transitions']['precision']:.3f}, "
                f"recall {best['transitions']['recall']:.3f}, "
                f"F1 {best['transitions']['f1']:.3f}, "
                f"GT {best['transitions']['gt_events']}, pred {best['transitions']['pred_events']}",
                "",
            ]
        )
    lines.extend(
        [
            "## Improvement Priorities",
            "",
            "1. Tune the operating confidence for transition recall, not only mAP. Lower thresholds recover more lamps but can create extra tracked transitions.",
            "2. Review false transition windows and frames with fewer or more than four predictions. These dominate downstream tracking reliability.",
            "3. Add temporal smoothing/hysteresis before declaring a red-white switch in production; single-frame class flicker should not become a transition.",
            "4. Add a qualitative review set around ground-truth transition frames and high-projection-distance frames.",
            "5. If model-side improvement is needed after review, train a larger YOLO26m or continue YOLO26n with hard examples rather than extending the same run blindly.",
            "",
        ]
    )
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--weights", type=Path, required=True)
    parser.add_argument(
        "--dataset-root",
        type=Path,
        default=REPO_ROOT / "data" / "datasets" / "papi_lamp_sequences",
    )
    parser.add_argument(
        "--split-root",
        type=Path,
        default=REPO_ROOT / "data" / "datasets" / "papi_lamp_sequences" / "yolo26n_combined",
    )
    parser.add_argument(
        "--airport-config",
        type=Path,
        default=REPO_ROOT / "configs" / "papi_edny.yaml",
    )
    parser.add_argument(
        "--projection-config",
        type=Path,
        default=REPO_ROOT / "configs" / "projection.yaml",
    )
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--splits", nargs="+", default=["val", "test"])
    parser.add_argument("--thresholds", nargs="+", type=float, default=[0.1, 0.25, 0.5, 0.75])
    parser.add_argument("--max-predictions-per-frame", type=int)
    parser.add_argument("--min-conf", type=float, default=0.001)
    parser.add_argument("--iou-threshold", type=float, default=0.5)
    parser.add_argument("--transition-frame-tolerance", type=int, default=1)
    parser.add_argument("--projection-max-distance-px", type=float, default=300.0)
    parser.add_argument("--imgsz", type=int, default=1280)
    parser.add_argument("--batch", type=int, default=4)
    parser.add_argument("--device", default="0")
    args = parser.parse_args()

    args.output_dir.mkdir(parents=True, exist_ok=True)
    metadata = _metadata_index(args.dataset_root)
    airport_config = _load_yaml(args.airport_config)
    projection_convention = _load_projection(args.projection_config)

    summary: dict[str, Any] = {
        "weights": str(args.weights.resolve()),
        "dataset_root": str(args.dataset_root.resolve()),
        "split_root": str(args.split_root.resolve()),
        "thresholds": args.thresholds,
        "iou_threshold": args.iou_threshold,
        "splits": {},
    }
    for split in args.splits:
        image_paths = _read_split(args.split_root / f"{split}.txt")
        predictions = _run_predictions(
            args.weights,
            image_paths,
            min_conf=args.min_conf,
            imgsz=args.imgsz,
            batch=args.batch,
            device=args.device,
        )
        threshold_results = [
            evaluate_threshold(
                split_name=split,
                image_paths=image_paths,
                metadata=metadata,
                predictions=predictions,
                airport_config=airport_config,
                projection_convention=projection_convention,
                threshold=threshold,
                max_predictions_per_frame=args.max_predictions_per_frame,
                iou_threshold=args.iou_threshold,
                transition_frame_tolerance=args.transition_frame_tolerance,
                projection_max_distance_px=args.projection_max_distance_px,
            )
            for threshold in args.thresholds
        ]
        summary["splits"][split] = {
            "images": len(image_paths),
            "threshold_results": threshold_results,
            "best_threshold": _best_by_threshold(threshold_results),
        }

    (args.output_dir / "sequence_eval_detailed.json").write_text(
        json.dumps(summary, indent=2) + "\n",
        encoding="utf-8",
    )
    _write_markdown(summary, args.output_dir / "sequence_eval_report.md")
    print(json.dumps(summary["splits"], indent=2))



## Run Full Evaluation

Set `RUN_EVALUATION = True` to regenerate both raw and top-4 reports. Keeping this off by default avoids re-running GPU inference whenever the notebook is opened.


In [ ]:
RUN_EVALUATION = False
THRESHOLDS = [0.05, 0.10, 0.25, 0.50, 0.75]

if RUN_EVALUATION:
    import sys
    for output_dir, max_pred in [(RAW_OUTPUT_DIR, None), (TOP4_OUTPUT_DIR, 4)]:
        argv = [
            "notebook",
            "--weights", str(WEIGHTS),
            "--dataset-root", str(DATASET_ROOT),
            "--split-root", str(SPLIT_ROOT),
            "--output-dir", str(output_dir),
            "--thresholds", *[str(value) for value in THRESHOLDS],
            "--batch", "4",
            "--device", "0",
        ]
        if max_pred is not None:
            argv += ["--max-predictions-per-frame", str(max_pred)]
        old_argv = sys.argv
        try:
            sys.argv = argv
            main()
        finally:
            sys.argv = old_argv


## Review Existing Results

This section reads the saved JSON reports and prints the operational threshold table.


In [ ]:
def print_threshold_table(name, path):
    data = json.loads((path / "sequence_eval_detailed.json").read_text(encoding="utf-8"))
    print(f"=== {name} ===")
    for split, split_summary in data["splits"].items():
        print(split)
        for row in split_summary["threshold_results"]:
            transitions = row["transitions"]
            print(
                "thr", row["threshold"],
                "detP", round(row["detection"]["precision_iou50_class_agnostic"], 3),
                "detR", round(row["detection"]["recall_iou50_class_agnostic"], 3),
                "cls", round(row["classification"]["accuracy_on_localized_matches"], 3),
                "trk", round(row["tracking"]["tracked_lamp_state_accuracy"], 3),
                "miss", row["tracking"]["missing_tracked_lamp_states"],
                "wrong", row["tracking"]["wrong_tracked_lamp_states"],
                "trans", f"{transitions['true_positive_events']}/{transitions['gt_events']} TP",
                "pred", transitions["pred_events"],
                "F1", round(transitions["f1"], 3),
                "exactTP", transitions["exact_frame"]["true_positive_events"],
                "four", round(row["detection"]["frames_with_four_predictions_rate"], 3),
            )
        best = split_summary["best_threshold"]
        print("best", best["threshold"], "tracking", round(best["tracking"]["tracked_lamp_state_accuracy"], 3), "transition_f1", round(best["transitions"]["f1"], 3))
    return data

raw_results = print_threshold_table("raw", RAW_OUTPUT_DIR)
top4_results = print_threshold_table("top4", TOP4_OUTPUT_DIR)


## Findings

- Detection/classification is usable: test classification on localized matches is about `0.994`.
- Raw low-threshold detections recover boxes but produce many extra detections, which destabilizes tracking.
- Top-4 PAPI geometry filtering improves tracked lamp-state accuracy to about `0.960` on validation and `0.952` on test at low confidence.
- Transition detection is still the weak point: best top-4 test transition F1 is about `0.174` with ±1-frame tolerance.
- Improve next with temporal smoothing/hysteresis, transition-window review, and hard-example fine-tuning before another blind training run.
